# TriPendulum-8 Colab Pro 一键训练

按 Colab 菜单 `Runtime -> Run all` 即可从安装依赖开始，自动完成环境检查、SAC/PPO 训练、动态诊断视频、评估、转换矩阵、视频渲染和结果打包。

重点：模型文件、checkpoint、TensorBoard logs、诊断视频和评估结果默认保存到 Google Drive，避免 Colab runtime 断开后丢失。

In [ ]:
# ===== 一键参数区：按需修改这里即可 =====
# 建议 Colab Pro 使用 Drive 持久化输出。
USE_GOOGLE_DRIVE = True

# 项目代码目录。可以在 Drive，也可以是你上传/解压到 /content 的目录。
PROJECT_DIR = "/content/TriPendulum-8"  # 例如 Drive: /content/drive/MyDrive/TriPendulum-8

# 所有训练产物都会保存到这里，不会随 Colab 关闭丢失。
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/TriPendulum-8-outputs"
RUN_NAME = "sac_colab_pro_run"  # 想新开一次实验就改名；想覆盖/继续同一目录就保持不变。

# Colab Pro 建议先用 SAC 主训练；PPO 可作为 baseline 另跑。
RUN_PPO = False
RUN_SAC = True

PPO_TIMESTEPS = 300_000
SAC_TIMESTEPS = 1_000_000

# 动态诊断视频：curriculum_worst 会自动录当前阶段最失败的目标。
DIAGNOSTIC_ENABLED = True
DIAGNOSTIC_EVAL_FREQ = 100_000
DIAGNOSTIC_MAX_STEPS = 800
DIAGNOSTIC_N_EVAL_EPISODES = 2
DIAGNOSTIC_MODE = "curriculum_worst"
DIAGNOSTIC_WORST_K = 2

# Curriculum 设置；如果你想直接全目标训练，设 CURRICULUM_ENABLED=False。
CURRICULUM_ENABLED = False
CURRICULUM_STAGE = 4

EVAL_EPISODES_PER_GOAL = 5
TRANSITION_TRIALS = 3
FINAL_RENDER_GOAL = "UUU"


In [ ]:
# ===== 挂载 Drive、进入项目目录、创建持久化输出目录 =====
import os, sys, textwrap, subprocess, json, pathlib, shutil
from pathlib import Path

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

if os.path.exists(PROJECT_DIR):
    os.chdir(PROJECT_DIR)
else:
    print(f"PROJECT_DIR 不存在: {PROJECT_DIR}")
    print("如果你是上传 zip，请先在左侧文件区解压，或把 PROJECT_DIR 改成实际路径。")

OUTPUT_ROOT = Path(DRIVE_OUTPUT_DIR) / RUN_NAME
CHECKPOINT_DIR = OUTPUT_ROOT / "checkpoints"
RUNS_DIR = OUTPUT_ROOT / "runs"
VIDEO_DIR = OUTPUT_ROOT / "videos"
DIAGNOSTIC_VIDEO_DIR = VIDEO_DIR / "diagnostics"
EVAL_DIR = OUTPUT_ROOT / "evaluation"
CONFIG_OUT_DIR = OUTPUT_ROOT / "configs"
ARCHIVE_DIR = OUTPUT_ROOT / "archives"

for d in [OUTPUT_ROOT, CHECKPOINT_DIR, RUNS_DIR, VIDEO_DIR, DIAGNOSTIC_VIDEO_DIR, EVAL_DIR, CONFIG_OUT_DIR, ARCHIVE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Current project directory:", os.getcwd())
print("Persistent output root:", OUTPUT_ROOT)
print("Checkpoint dir:", CHECKPOINT_DIR)
print("TensorBoard dir:", RUNS_DIR)
print("Diagnostic video dir:", DIAGNOSTIC_VIDEO_DIR)
print("Files:", os.listdir('.')[:20])


In [ ]:
# ===== 安装依赖 =====
!python -m pip install -U pip
!pip install -r requirements.txt


In [ ]:
# ===== 检查 GPU / MuJoCo / SB3 =====
!nvidia-smi || true

import mujoco, gymnasium, stable_baselines3
print('MuJoCo:', mujoco.__version__)
print('Gymnasium:', gymnasium.__version__)
print('Stable-Baselines3:', stable_baselines3.__version__)


In [ ]:
# ===== 写入 Colab Pro 训练配置覆盖文件 =====
import yaml, os, shutil

with open('configs/sac.yaml', 'r') as f:
    sac_cfg = yaml.safe_load(f)
with open('configs/ppo.yaml', 'r') as f:
    ppo_cfg = yaml.safe_load(f)

# 这里生成独立 Colab 配置，不改原始 configs/default.yaml。
# 关键：所有输出目录都指向 Google Drive 下的 OUTPUT_ROOT。
colab_common = {
    'curriculum': {
        'enabled': CURRICULUM_ENABLED,
        'stage': CURRICULUM_STAGE,
    },
    'paths': {
        'checkpoint_dir': str(CHECKPOINT_DIR),
        'tensorboard_dir': str(RUNS_DIR),
    },
    'diagnostic_video': {
        'enabled': DIAGNOSTIC_ENABLED,
        'eval_freq': DIAGNOSTIC_EVAL_FREQ,
        'max_steps': DIAGNOSTIC_MAX_STEPS,
        'n_eval_episodes': DIAGNOSTIC_N_EVAL_EPISODES,
        'save_dir': str(DIAGNOSTIC_VIDEO_DIR),
        'fps': 30,
        'mode': DIAGNOSTIC_MODE,
        'worst_k': DIAGNOSTIC_WORST_K,
        'fallback_goals': ['DDD', 'UUU'],
    },
    'eval': {
        'enabled': False,
        'eval_freq': 100_000,
        'n_eval_episodes': 5,
        'best_model_save_path': str(CHECKPOINT_DIR),
        'log_path': str(RUNS_DIR / 'eval'),
    },
}

for cfg, algo_name, total_steps in [(sac_cfg, 'sac', SAC_TIMESTEPS), (ppo_cfg, 'ppo', PPO_TIMESTEPS)]:
    cfg.update(colab_common)
    cfg.setdefault('algorithm', {})['total_timesteps'] = int(total_steps)
    local_cfg_path = f'configs/colab_{algo_name}.yaml'
    drive_cfg_path = CONFIG_OUT_DIR / f'colab_{algo_name}.yaml'
    with open(local_cfg_path, 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    shutil.copy(local_cfg_path, drive_cfg_path)

SAC_MODEL_PATH = str(CHECKPOINT_DIR / 'sac_best.zip')
PPO_MODEL_PATH = str(CHECKPOINT_DIR / 'ppo_colab_final.zip')
MODEL_PATH = SAC_MODEL_PATH if RUN_SAC else PPO_MODEL_PATH

print('SAC model path:', SAC_MODEL_PATH)
print('PPO model path:', PPO_MODEL_PATH)
print('Config copy in Drive:', CONFIG_OUT_DIR)
print(open('configs/colab_sac.yaml').read())


In [ ]:
# ===== 环境 smoke test：随机动作、角度转换、8 个绝对目标 =====
import numpy as np
from envs.tripendulum_env import TriPendulumGoalEnv
from envs.goals import GOAL_NAMES, GOAL_ABS_ANGLES, GOAL_BINARY
from utils.angle_utils import relative_to_absolute

env = TriPendulumGoalEnv()
obs, info = env.reset(goal='UUU')
print('obs shape:', obs.shape)
print('reset info:', {k: info[k] for k in ['goal_name', 'x', 'x_max']})

for i in range(10):
    obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
    print(i, 'reward=', round(reward, 3), 'goal=', info['goal_name'], 'x=', round(info['x'], 3), 'pose=', round(info.get('r_pose', 0), 3))
    if terminated or truncated:
        obs, info = env.reset()

env.close()

q = np.array([0.1, 0.2, -0.3])
print('relative q:', q)
print('absolute theta:', relative_to_absolute(q))
for name in GOAL_NAMES:
    print(name, 'binary=', GOAL_BINARY[name], 'abs=', GOAL_ABS_ANGLES[name])


In [ ]:
# ===== 启动 TensorBoard（读取 Drive 中的持久化 logs） =====
TB_LOGDIR = str(RUNS_DIR)
%load_ext tensorboard
%tensorboard --logdir $TB_LOGDIR


In [ ]:
# ===== PPO baseline 训练（默认关闭，checkpoint 保存到 Drive） =====
if RUN_PPO:
    !python training/train_ppo.py --config configs/colab_ppo.yaml --total-timesteps {PPO_TIMESTEPS} --save-path {PPO_MODEL_PATH}
else:
    print('RUN_PPO=False，跳过 PPO baseline。')


In [ ]:
# ===== SAC 主训练（模型、checkpoint、诊断视频全部保存到 Drive） =====
if RUN_SAC:
    !python training/train_sac.py --config configs/colab_sac.yaml --total-timesteps {SAC_TIMESTEPS} --save-path {SAC_MODEL_PATH}
else:
    print('RUN_SAC=False，跳过 SAC。')


In [ ]:
# ===== 查看 Drive 中的动态诊断报告和视频 =====
import glob, json, os
from IPython.display import Video, display

reports = sorted(glob.glob(str(DIAGNOSTIC_VIDEO_DIR / 'diagnostic_step_*.json')))
print('diagnostic reports:', reports[-5:])
if reports:
    latest_report = reports[-1]
    with open(latest_report) as f:
        report = json.load(f)
    print('latest report:', latest_report)
    print(json.dumps(report, indent=2)[:4000])

videos = sorted(glob.glob(str(DIAGNOSTIC_VIDEO_DIR / '*.mp4')))
print('diagnostic videos:', videos[-5:])
if videos:
    display(Video(videos[-1], embed=True))


In [ ]:
# ===== 8 个目标评估（输出保存到 Drive） =====
EVAL_RESULTS_CSV = str(EVAL_DIR / 'evaluation_results.csv')
!python evaluation/evaluate.py --model {MODEL_PATH} --episodes-per-goal {EVAL_EPISODES_PER_GOAL} --output {EVAL_RESULTS_CSV}

import pandas as pd
pd.read_csv(EVAL_RESULTS_CSV)


In [ ]:
# ===== 生成 8x8 姿态切换热力图（CSV/PNG 保存到 Drive） =====
TRANSITION_CSV = str(EVAL_DIR / 'transition_success_matrix.csv')
TRANSITION_HEATMAP = str(EVAL_DIR / 'transition_success_heatmap.png')
!python evaluation/transition_matrix.py --model {MODEL_PATH} --trials {TRANSITION_TRIALS} --csv {TRANSITION_CSV} --heatmap {TRANSITION_HEATMAP}

from IPython.display import Image, display
if os.path.exists(TRANSITION_HEATMAP):
    display(Image(TRANSITION_HEATMAP))


In [ ]:
# ===== 渲染最终指定 goal 视频（保存到 Drive） =====
FINAL_VIDEO_PATH = str(VIDEO_DIR / f'final_{FINAL_RENDER_GOAL}.mp4')
!python evaluation/render_video.py --model {MODEL_PATH} --goal {FINAL_RENDER_GOAL} --output {FINAL_VIDEO_PATH}

from IPython.display import Video, display
if os.path.exists(FINAL_VIDEO_PATH):
    display(Video(FINAL_VIDEO_PATH, embed=True))


In [ ]:
# ===== 打包 Drive 中的训练结果 =====
import shutil, os, glob
archive_base = Path(DRIVE_OUTPUT_DIR) / f'{RUN_NAME}_results'
zip_path = str(archive_base) + '.zip'
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    str(archive_base),
    'zip',
    root_dir=str(OUTPUT_ROOT),
    base_dir='.',
)
print('Created:', zip_path)
print('Size MB:', os.path.getsize(zip_path) / 1024 / 1024)
print('All persistent outputs are under:', OUTPUT_ROOT)
